This notebook uses Simulated Annealing (SA) to further optimize tree placements and reduce the total score.
💡 How it works

We take an existing high-quality solution and try to "squeeze" it even further:

    Perturb: We run the bbox3 solver with random parameters to find a new configuration for the trees.

    Evaluate: If the new score is better, we keep it.

    Escape: If the score is slightly worse, we might still accept it (based on "temperature"). This helps the algorithm avoid getting stuck in a local optimum.

🤝 Acknowledgments

This notebook is based on the great work of the Kaggle community:

    - https://www.kaggle.com/code/daniilkrizhanovskyi/new-year-same-old-bbox;
     
    - https://www.kaggle.com/code/datafad/a-bit-better;
    
    - https://www.kaggle.com/code/chistyakov/squeezing-bbox-solutio;

    - https://www.kaggle.com/code/datafad/new-year-same-old-bbox;

    - https://www.kaggle.com/code/saspav/santa-submission

    - https://www.kaggle.com/code/yongsukprasertsuk/santa2025-bbox-optimization

    - https://www.kaggle.com/code/chistyakov/squeezing-bbox-solution

    - https://www.kaggle.com/code/datafad/single-group-optimizer-by-manoj

    - https://www.kaggle.com/code/jazivxt/why-not

    - /kaggle/input/k/datafad/new-year-same-old-bbox/submission.csv

In [ ]:
# ==============================
# AUTO SQUEEZE bbox3 v3
# - detect -i/-o if supported
# - discard "no-change" runs
# - calibrate T0 only on changed candidates
# - plateau restart to best
# - no temp leaks
# ==============================

import os, shutil, subprocess, random, math, tempfile, time, hashlib
import numpy as np
import pandas as pd
from numba import njit

from decimal import Decimal, getcontext
from shapely.geometry import Polygon
from shapely import affinity
from shapely.strtree import STRtree

# ------------------------------
# CONFIG
# ------------------------------
START_SUB  = "/kaggle/input/k/datafad/new-year-same-old-bbox/submission.csv"
START_BBOX = "/kaggle/input/k/datafad/new-year-same-old-bbox/bbox3"

WORK_SUB = "submission.csv"
BBOX     = "./bbox3"

MAX_SECONDS = 60 * 60  # 1h

# bbox3 参数范围
N_RANGE_WIDE   = (50, 1000)
R_RANGE_WIDE   = (10, 150)
N_RANGE_NARROW = (50, 400)
R_RANGE_NARROW = (10, 60)

# SA 参数
USE_SA = True
AUTO_CALIBRATE_T0 = True
CALIB_TRIALS = 16
T0_FALLBACK = 5e-3
T_MIN = 1e-6
ALPHA = 0.995

# overlap 校验策略
VALIDATE_BEST_ALWAYS = True     # best 更新必校验（慢但安全）
VALIDATE_CURRENT_EVERY = 30     # current 每接受 N 次抽查一次（防污染，别太小）

# plateau 重启
PLATEAU_ITERS = 400             # 连续这么多 iter best 不变 -> current 回到 best

# 严格几何精度
getcontext().prec = 25
SCALE = Decimal("1e18")

TREE_PTS = np.array([
    (0,0.8),(0.125,0.5),(0.0625,0.5),(0.2,0.25),(0.1,0.25),(0.35,0),
    (0.075,0),(0.075,-0.2),(-0.075,-0.2),(-0.075,0),(-0.35,0),
    (-0.1,0.25),(-0.2,0.25),(-0.0625,0.5),(-0.125,0.5)
], dtype=np.float64)
TREE_TX = TREE_PTS[:, 0].astype(np.float64)
TREE_TY = TREE_PTS[:, 1].astype(np.float64)

# ------------------------------
# INIT
# ------------------------------
if not os.path.exists(WORK_SUB):
    shutil.copy(START_SUB, WORK_SUB)
    print("✅ Initial submission copied -> ./submission.csv")

if not os.path.exists(BBOX):
    shutil.copy(START_BBOX, BBOX)
os.chmod(BBOX, 0o755)
print("✅ bbox3 ready")

BBOX_ABS = os.path.abspath(BBOX)

def sha1_file(path: str) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# ------------------------------
# detect bbox3 supports -i/-o
# ------------------------------
def detect_io_args() -> bool:
    try:
        p = subprocess.run([BBOX_ABS, "-h"], capture_output=True, text=True, check=False)
        out = (p.stdout or "") + "\n" + (p.stderr or "")
        out = out.lower()
        return ("-i" in out) and ("-o" in out)
    except:
        return False

USE_IO_ARGS = detect_io_args()
print(f"ℹ️ bbox3 supports -i/-o: {USE_IO_ARGS}")

def bbox3_cmd(n: int, r: int):
    if USE_IO_ARGS:
        return [BBOX_ABS, "-i", "submission.csv", "-o", "submission.csv", "-n", str(n), "-r", str(r)]
    else:
        # 常见 bbox3 直接在 cwd 读写 submission.csv
        return [BBOX_ABS, "-n", str(n), "-r", str(r)]

# ------------------------------
# FAST SCORE (numba)
# ------------------------------
def _parse_submission_fast(path: str):
    df = pd.read_csv(
        path,
        usecols=["id", "x", "y", "deg"],
        dtype={"id": "string", "x": "string", "y": "string", "deg": "string"},
        engine="c"
    )
    grp = df["id"].str.slice(0, 3).astype(np.int16).to_numpy()
    x   = df["x"].str.slice(1).astype(np.float64).to_numpy()
    y   = df["y"].str.slice(1).astype(np.float64).to_numpy()
    deg = df["deg"].str.slice(1).astype(np.float64).to_numpy()
    return grp, x, y, deg

@njit
def eval_fast_numba(grp, x, y, deg, tx, ty):
    total = 0.0
    for n in range(1, 201):
        mnx = 1e300; mny = 1e300
        mxx = -1e300; mxy = -1e300
        found = False

        for i in range(grp.size):
            if grp[i] != n:
                continue
            found = True
            r = deg[i] * math.pi / 180.0
            c = math.cos(r); s = math.sin(r)
            xi = x[i]; yi = y[i]
            for j in range(tx.size):
                X = c*tx[j] - s*ty[j] + xi
                Y = s*tx[j] + c*ty[j] + yi
                if X < mnx: mnx = X
                if X > mxx: mxx = X
                if Y < mny: mny = Y
                if Y > mxy: mxy = Y

        if found:
            side = mxx - mnx
            h = mxy - mny
            if h > side: side = h
            total += (side * side) / n
    return total

def eval_fast(path: str) -> float:
    grp, x, y, deg = _parse_submission_fast(path)
    return float(eval_fast_numba(grp, x, y, deg, TREE_TX, TREE_TY))

# numba warmup
_ = eval_fast(WORK_SUB)

# ------------------------------
# STRICT overlap validation (slow)
# ------------------------------
class ChristmasTreeStrict:
    def __init__(self, x, y, d):
        self.x = Decimal(str(x))
        self.y = Decimal(str(y))
        self.d = Decimal(str(d))

        pts = [(Decimal(str(px)), Decimal(str(py))) for (px, py) in TREE_PTS.tolist()]
        poly = Polygon([(float(px * SCALE), float(py * SCALE)) for px, py in pts])
        poly = affinity.rotate(poly, float(self.d), origin=(0, 0))
        self.p = affinity.translate(poly, float(self.x * SCALE), float(self.y * SCALE))

def _strtree_query_indices(tree, poly, polys):
    res = tree.query(poly)
    if len(res) == 0:
        return []
    first = res[0]
    if isinstance(first, (int, np.integer)):
        return res
    mp = {id(g): i for i, g in enumerate(polys)}
    return [mp.get(id(g), -1) for g in res]

def has_overlap_group(df_group: pd.DataFrame) -> bool:
    trees = []
    for _, r in df_group.iterrows():
        x = float(str(r["x"])[1:])
        y = float(str(r["y"])[1:])
        d = float(str(r["deg"])[1:])
        trees.append(ChristmasTreeStrict(x, y, d))

    if len(trees) < 2:
        return False

    polys = [t.p for t in trees]
    idx = STRtree(polys)

    for i, p in enumerate(polys):
        js = _strtree_query_indices(idx, p, polys)
        for j in js:
            if j < 0 or j == i:
                continue
            if p.intersects(polys[j]) and not p.touches(polys[j]):
                return True
    return False

def validate_no_overlap(path: str) -> bool:
    df = pd.read_csv(
        path,
        usecols=["id", "x", "y", "deg"],
        dtype={"id": "string", "x": "string", "y": "string", "deg": "string"},
        engine="c"
    )
    df["_grp"] = df["id"].str.slice(0, 3).astype(int)
    for n in range(1, 201):
        g = df[df["_grp"] == n]
        if g.empty:
            continue
        if has_overlap_group(g):
            return False
    return True

# ------------------------------
# bbox3 runner (tmp dir) + "changed?" check
# ------------------------------
def run_bbox3_tmp(start_sub_path: str, n: int, r: int):
    tmp = tempfile.mkdtemp(prefix="bbox3tmp_")
    sub_path = os.path.join(tmp, "submission.csv")
    shutil.copy(start_sub_path, sub_path)

    before = sha1_file(sub_path)
    p = subprocess.run(
        bbox3_cmd(n, r),
        cwd=tmp,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False
    )

    after = sha1_file(sub_path)
    changed = (before != after)
    return tmp, sub_path, p.returncode, changed

def cleanup_tmp(tmpdir: str):
    shutil.rmtree(tmpdir, ignore_errors=True)

def save_candidate(tmp_csv_path: str) -> str:
    stable = tempfile.NamedTemporaryFile(delete=False, suffix=".csv")
    shutil.copy(tmp_csv_path, stable.name)
    return stable.name

# ------------------------------
# parameter sampling
# ------------------------------
def sample_params(progress: float, best_params):
    if progress < 0.3:
        n_min, n_max = N_RANGE_WIDE
        r_min, r_max = R_RANGE_WIDE
    else:
        n_min, n_max = N_RANGE_NARROW
        r_min, r_max = R_RANGE_NARROW

    # exploit around best 70%
    if best_params is not None and random.random() < 0.7:
        bn, br = best_params
        n = int(np.clip(np.random.normal(bn, 80), n_min, n_max))
        r = int(np.clip(np.random.normal(br, 15), r_min, r_max))
        return int(np.clip(n, n_min, n_max)), int(np.clip(r, r_min, r_max))

    return random.randint(n_min, n_max), random.randint(r_min, r_max)

# ------------------------------
# T0 calibration (ONLY on changed candidates)
# ------------------------------
def calibrate_T0(current_sub: str, trials: int):
    base = eval_fast(current_sub)
    deltas = []
    attempts = 0
    while len(deltas) < trials and attempts < trials * 5:
        attempts += 1
        n = random.randint(*N_RANGE_WIDE)
        r = random.randint(*R_RANGE_WIDE)
        tmpdir, tmpcsv, rc, changed = run_bbox3_tmp(current_sub, n, r)
        try:
            if rc != 0 or (not changed):
                continue
            sc = eval_fast(tmpcsv)
            deltas.append(abs(sc - base))
        finally:
            cleanup_tmp(tmpdir)

    if len(deltas) == 0:
        return T0_FALLBACK

    med = float(np.median(np.array(deltas)))
    # 给一个相对宽松的初温（可接受一些“略差”）
    return max(med * 8.0, 1e-6)

# ------------------------------
# MAIN LOOP
# ------------------------------
random.seed(0)
np.random.seed(0)

start_time = time.time()
deadline = start_time + MAX_SECONDS

current_sub = WORK_SUB
current_score = eval_fast(current_sub)
best_sub = WORK_SUB
best_score = current_score
best_params = None

if USE_SA and AUTO_CALIBRATE_T0:
    T0 = calibrate_T0(current_sub, CALIB_TRIALS)
else:
    T0 = T0_FALLBACK

T = T0
print(f"🔥 Start fast-score: {current_score:.12f} | T0={T0:.2e}")

iters = 0
accepted = 0
improved = 0
validated = 0
skipped_nochange = 0
skipped_fail = 0
plateau = 0
rejected_invalid_current = 0

while time.time() < deadline:
    iters += 1
    progress = (time.time() - start_time) / MAX_SECONDS

    n, r = sample_params(progress, best_params)

    tmpdir, tmpcsv, rc, changed = run_bbox3_tmp(current_sub, n=n, r=r)
    try:
        if rc != 0:
            skipped_fail += 1
            continue
        if not changed:
            skipped_nochange += 1
            continue

        cand_score = eval_fast(tmpcsv)
        delta = cand_score - current_score

        if USE_SA:
            if delta < 0:
                accept = True
            else:
                accept = (random.random() < math.exp(-delta / max(T, 1e-12)))
        else:
            accept = (delta < 0)

        if accept:
            accepted += 1

            # 抽查 current overlap（防污染）
            if (accepted % VALIDATE_CURRENT_EVERY) == 0:
                if not validate_no_overlap(tmpcsv):
                    accept = False
                    rejected_invalid_current += 1

        # best 更新（严格校验）
        if cand_score < best_score:
            ok = True
            if VALIDATE_BEST_ALWAYS:
                validated += 1
                ok = validate_no_overlap(tmpcsv)
            if ok:
                new_best = save_candidate(tmpcsv)

                old_best = best_sub
                best_sub = new_best
                best_score = cand_score
                best_params = (n, r)
                improved += 1
                plateau = 0

                if old_best not in (WORK_SUB, current_sub) and os.path.exists(old_best):
                    try: os.remove(old_best)
                    except: pass

                print(f"🏆 BEST @ iter {iters}: {best_score:.12f} (n={n}, r={r})")
        else:
            plateau += 1

        # 更新 current（接受时）
        if accept:
            new_current = save_candidate(tmpcsv)
            old_current = current_sub
            current_sub = new_current
            current_score = cand_score

            if old_current not in (WORK_SUB, best_sub) and os.path.exists(old_current):
                try: os.remove(old_current)
                except: pass

        # plateau restart：很久没进步 -> current 回 best
        if plateau >= PLATEAU_ITERS and os.path.abspath(current_sub) != os.path.abspath(best_sub):
            # 把 current 重置到 best，避免在差区域乱走
            current_sub = best_sub
            current_score = best_score
            plateau = 0
            print(f"🔁 Plateau restart -> current = best (iter {iters})")

        # 温度衰减
        if USE_SA:
            T = max(T * ALPHA, T_MIN)

        # 日志
        if iters % 100 == 0:
            elapsed = time.time() - start_time
            print(
                f"Iter {iters:6d} | {elapsed:7.1f}s | T={T:.2e} | "
                f"cur={current_score:.12f} | best={best_score:.12f} | "
                f"acc={accepted} | imp={improved} | val={validated} | "
                f"skipNoChg={skipped_nochange} | skipFail={skipped_fail} | rejInvCur={rejected_invalid_current}"
            )

    finally:
        cleanup_tmp(tmpdir)

# finalize
if os.path.abspath(best_sub) != os.path.abspath(WORK_SUB):
    shutil.copy(best_sub, WORK_SUB)

print("\n✅ FINAL BEST FAST-SCORE:", best_score)
print("📄 submission.csv READY at /kaggle/working/submission.csv")
print(f"Stats: iters={iters}, improved={improved}, accepted={accepted}, "
      f"skip_nochange={skipped_nochange}, skip_fail={skipped_fail}")
